# Day 3 v2 — Section 5: MAE Optimization + Hypothesis Test

**Mục tiêu:** Cải thiện MAE từ baseline 108.8k (LGB char_wb 50K) xuống thấp nhất có thể.

**Sections:**
- 5A. LGB + char_wb TF-IDF — full 269K train (MSE objective)
- 5B. LGB + char_wb TF-IDF — full 269K train (MAE objective = regression_l1)
- 5C. Blend 5A + 5B — optimize weight trên val set
- 5D. RandomForest + BoW 2000 — hypothesis test (so sánh với English day3)
- 5E. XGBoost + BoW 2000 — hypothesis test (so sánh với English day3)
- 5F. LGB + CountVectorizer word (1,2) 50K — full 269K (MSE objective)
- 5G. LGB + CountVectorizer word (1,2) 50K — full 269K (MAE objective)
- 5H. Blend 5F + 5G — optimize weight trên val set

**Baseline (Section 0-4):** Best = 3B. LGB+char_wb 50K subset → MAE=108.8k, R²=59.6%

**Config máy:** i5-14600KF 18 cores / i7-12700K 20 cores | RAM 28GB | CUDA

In [ ]:
import sys, json
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestRegressor

sys.path.append("..")
from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"train={len(train):,} | val={len(val):,} | test={len(test):,}")

documents   = [item.summary for item in train]
prices      = np.array([item.price for item in train], dtype=np.float32)
docs_val    = [item.summary for item in val]
prices_val  = np.array([item.price for item in val], dtype=np.float32)

In [ ]:
# Ket qua Section 0-4 (hard-coded tu session truoc)
results = {
    "2b. LR + BoW":              {"mae": 131.7, "mse": 31955, "r2": 0.443},
    "2c. Ridge + TF-IDF char_wb":{"mae": 112.1, "mse": 23401, "r2": 0.592},
    "3A. Bench BoW + LGB":       {"mae": 120.8, "mse": 28226, "r2": 0.508},
    "3B. Bench char_wb + LGB":   {"mae": 108.8, "mse": 23173, "r2": 0.596},
    "3C. Bench Underthesea+LGB": {"mae": 118.0, "mse": 28239, "r2": 0.508},
    "4a. RandomForest (15K)": {"mae": 129.7, "mse": None, "r2": 0.423},
    "4b. XGBoost (269K)": {"mae": 125.2, "mse": None, "r2": 0.435},
}
print("Baseline results loaded:", len(results), "models")

## Section 5A — LGB full 269K (MSE objective)

Refit TF-IDF char_wb trên full 269K (thay vì 50K subset).
num_leaves=63 (tang tu 31 vi co nhieu data hon).

In [ ]:
# Fit TF-IDF char_wb tren full 269K — dung chung cho 5A va 5B
print("Fitting TF-IDF char_wb on 269K docs...")
vec_main = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)
X_main = vec_main.fit_transform(documents)
print(f"Feature matrix: {X_main.shape}")

In [ ]:
# 5A. LGB MSE — full 269K
# num_leaves=63 de tang capacity voi 5x data so voi benchmark (50K)
print("[5A] Training LGB (MSE, full 269K)...")
lgb_mse = lgb.LGBMRegressor(
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    min_child_samples=30,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
lgb_mse.fit(X_main, prices)
print("Training done.")

def lgb_mse_pricer(item):
    x = vec_main.transform([item.summary])
    return max(5, lgb_mse.predict(x)[0])

print("Evaluating [5A]:")
results["5A. LGB MSE (269K)"] = evaluate(lgb_mse_pricer, test)

## Section 5B — LGB full 269K (MAE objective)

objective='regression_l1' toi uu MAE truc tiep (thay vi MSE).
Dung lai X_main da fit o 5A.

In [ ]:
# 5B. LGB MAE — full 269K, objective='regression_l1'
print("[5B] Training LGB (MAE objective, full 269K)...")
lgb_mae = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    min_child_samples=30,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
lgb_mae.fit(X_main, prices)
print("Training done.")

def lgb_mae_pricer(item):
    x = vec_main.transform([item.summary])
    return max(5, lgb_mae.predict(x)[0])

print("Evaluating [5B]:")
results["5B. LGB MAE obj (269K)"] = evaluate(lgb_mae_pricer, test)

## Section 5C — Blend 5A + 5B

Tim blend weight toi uu tren val set (3,926 items), sau do evaluate tren test set.

In [ ]:
# 5C. Blend 5A + 5B — optimize weight tren val set
X_val = vec_main.transform(docs_val)
pred_mse_val = lgb_mse.predict(X_val)
pred_mae_val = lgb_mae.predict(X_val)

best_w, best_val_mae = 0.5, float("inf")
for w in np.arange(0.0, 1.01, 0.05):
    blended = w * pred_mse_val + (1 - w) * pred_mae_val
    blended = np.clip(blended, 5, 1000)
    val_mae = np.mean(np.abs(blended - prices_val))
    if val_mae < best_val_mae:
        best_val_mae, best_w = val_mae, w

print(f"Best blend: {best_w:.2f}*5A + {1-best_w:.2f}*5B | val MAE={best_val_mae:.1f}k")

def lgb_blend_pricer(item):
    x = vec_main.transform([item.summary])
    p_mse = lgb_mse.predict(x)[0]
    p_mae = lgb_mae.predict(x)[0]
    return max(5, best_w * p_mse + (1 - best_w) * p_mae)

print("Evaluating [5C] on test:")
results["5C. Blend 5A+5B"] = evaluate(lgb_blend_pricer, test)

## Section 5D — RandomForest + CountVect word (1,2) 50K (Hypothesis Test)

Gia thuyet: RF thua o Section 4 vi TF-IDF 100K features qua cao chieu.
Neu dung CountVect word (1,2) 50K (thay vi 2000), RF co the tot hon.

Config: CountVectorizer(analyzer='word', ngram_range=(1,2), max_features=50_000)
Khong dung stop_words — tu dinh gia ('moi', 'chinh hang') quan trong.

In [ ]:
vec_bow_hyp = CountVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=50_000,
)
X_bow = vec_bow_hyp.fit_transform(documents).astype(np.float32)
print(f"  Count matrix: {X_bow.shape}")

In [ ]:
# 5D. RF + CountVect word (1,2) 50K — hypothesis test
print("[5D] Fitting CountVect word (1,2) 50K...")


subset = 15_000
print(f"[5D] Training RF (100 trees, {subset:,} subset)...")
rf_bow = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_bow.fit(X_bow[:subset], prices[:subset])
print("Training done.")

def rf_bow_pricer(item):
    x = vec_bow_hyp.transform([item.summary]).astype(np.float32)
    return max(5, rf_bow.predict(x)[0])

print("Evaluating [5D]:")
results["5D. RF + CountVect word 50K"] = evaluate(rf_bow_pricer, test)

## Section 5E — XGBoost + CountVect word (1,2) 50K (Hypothesis Test)

Gia thuyet: XGBoost thua o Section 4 vi TF-IDF 100K features.
CountVect word (1,2) 50K se cho ket qua tot hon.
Dung lai vec_bow_hyp va X_bow tu 5D.

In [ ]:
# 5E. XGBoost + CountVect word (1,2) 50K — hypothesis test
# Dung lai vec_bow_hyp va X_bow tu 5D
print("[5E] Training XGBoost (CountVect word 50K, full 269K)...")
np.random.seed(42)
xgb_bow = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)
xgb_bow.fit(X_bow, prices)
print("Training done.")

def xgb_bow_pricer(item):
    x = vec_bow_hyp.transform([item.summary]).astype(np.float32)
    return max(5, xgb_bow.predict(x)[0])

print("Evaluating [5E]:")
results["5E. XGBoost + CountVect word 50K"] = evaluate(xgb_bow_pricer, test)

## Section 5F — LGB + CountVectorizer word (1,2) 50K (MSE)

So sanh voi 5A: thay TF-IDF char_wb bang CountVectorizer word-level.
- CountVect word (1,2) 50K: raw count, khong IDF weighting
- TF-IDF char_wb (2,4) 100K: co IDF weighting, char-level features
Ket qua cho thay IDF weighting va char ngrams co giup ich khong.

In [ ]:
# Fit CountVectorizer word-level (1,2) 50K — dung chung cho 5F va 5G
print("Fitting CountVectorizer word (1,2) 50K on 269K docs...")
vec_count = CountVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=50_000,
)
X_count = vec_count.fit_transform(documents)
print(f"  Count matrix: {X_count.shape}")

# 5F. LGB MSE — CountVectorizer word (1,2) 50K, full 269K
print("[5F] Training LGB (MSE, CountVect word 50K)...")
lgb_count_mse = lgb.LGBMRegressor(
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    min_child_samples=30,
    n_jobs=-1,
    random_state=42,
)
lgb_count_mse.fit(X_count, prices)
print("Training done.")

def lgb_count_mse_pricer(item):
    x = vec_count.transform([item.summary])
    return max(5, lgb_count_mse.predict(x)[0])

print("Evaluating [5F]:")
results["5F. LGB + CountVect word 50K (MSE)"] = evaluate(lgb_count_mse_pricer, test)

## Section 5G — LGB + CountVectorizer word (1,2) 50K (MAE objective)

objective='regression_l1' toi uu MAE truc tiep.
Dung lai vec_count va X_count da fit o 5F.

In [ ]:
# 5G. LGB MAE — CountVectorizer word (1,2) 50K, full 269K
# Dung lai vec_count va X_count tu 5F
print("[5G] Training LGB (MAE objective, CountVect word 50K)...")
lgb_count_mae = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    min_child_samples=30,
    n_jobs=-1,
    random_state=42,
)
lgb_count_mae.fit(X_count, prices)
print("Training done.")

def lgb_count_mae_pricer(item):
    x = vec_count.transform([item.summary])
    return max(5, lgb_count_mae.predict(x)[0])

print("Evaluating [5G]:")
results["5G. LGB + CountVect word 50K (MAE)"] = evaluate(lgb_count_mae_pricer, test)

## Section 5H — Blend 5F + 5G

Tim blend weight toi uu tren val set, sau do evaluate tren test set.
Mirror cua 5C (Blend 5A+5B) nhung dung CountVectorizer thay TF-IDF.

In [ ]:
# 5H. Blend 5F + 5G — optimize weight tren val set
X_count_val = vec_count.transform(docs_val)
pred_count_mse_val = lgb_count_mse.predict(X_count_val)
pred_count_mae_val = lgb_count_mae.predict(X_count_val)

best_w_h, best_val_mae_h = 0.5, float("inf")
for w in np.arange(0.0, 1.01, 0.05):
    blended = w * pred_count_mse_val + (1 - w) * pred_count_mae_val
    blended = np.clip(blended, 5, 1000)
    val_mae = np.mean(np.abs(blended - prices_val))
    if val_mae < best_val_mae_h:
        best_val_mae_h, best_w_h = val_mae, w

print(f"Best blend: {best_w_h:.2f}*5F + {1-best_w_h:.2f}*5G | val MAE={best_val_mae_h:.1f}k")

def lgb_count_blend_pricer(item):
    x = vec_count.transform([item.summary])
    p_mse = lgb_count_mse.predict(x)[0]
    p_mae = lgb_count_mae.predict(x)[0]
    return max(5, best_w_h * p_mse + (1 - best_w_h) * p_mae)

print("Evaluating [5H] on test:")
results["5H. Blend 5F+5G (CountVect)"] = evaluate(lgb_count_blend_pricer, test)

## Tong hop ket qua Section 0-5H

In [ ]:
# In bang so sanh day du
print("=" * 70)
print(f"{'MODEL':<35} {'MAE':>8}  {'R2':>7}")
print("=" * 70)

# Sort by MAE
sorted_results = sorted(
    [(k, v) for k, v in results.items() if v.get("mae") is not None],
    key=lambda x: x[1]["mae"]
)
for name, m in sorted_results:
    r2 = f"{m['r2']*100:.1f}%" if m.get('r2') is not None else "N/A"
    print(f"{name:<35} {m['mae']:>6.1f}k  {r2:>7}")
print("=" * 70)

In [ ]:
# Save full results
out_path = "day3_v2_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved {len(results)} models to {out_path}")